# Data Check

## Imports and paths

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Find the repo root (the folder that contains ReadMe.md)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ReadMe.md").exists())
RAW_PATH   = ROOT / "Data/DataMainTable/diabetic_data.csv"
CLEAN_PATH = ROOT / "Data/cleaned_diabetic_data.csv"
FIG = ROOT / "Paper/figures"; FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", context="notebook")
print("Repo root:", ROOT) # use the training split

Repo root: C:\Users\JanAr\IdeaProjects\Data-Analytics-diabetes-readmission\Data-Analytics-diabetes-readmission


## Load the data

In [43]:
# Only "?" is missing. keep_default_na=False stops "None" (= not tested) becoming NaN.
raw = pd.read_csv(RAW_PATH, na_values="?", keep_default_na=False,
                  dtype={"payer_code": str})
df = pd.read_csv(CLEAN_PATH)
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)

print(f"Raw:     {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Cleaned: {df.shape[0]:,} rows × {df.shape[1]} columns (34 + readmitted_30d)")

Raw:     101,766 rows × 50 columns
Cleaned: 69,987 rows × 35 columns (34 + readmitted_30d)


## Unit of analysis
This cell counts the patients and their encounters. Many patients have more than one encounter, so rows in the raw data are not independent. The cleaned data must have one row for each patient.

In [44]:
visits = raw.groupby("patient_nbr").size()
print(f"Unique patients:                 {visits.size:,}")
print(f"Patients with >1 encounter:      {(visits > 1).sum():,}")
print(f"Maximum encounters for 1 patient: {visits.max()}")
print(f"Cleaned rows per patient (max):  {df.groupby('patient_nbr').size().max()}")

Unique patients:                 71,518
Patients with >1 encounter:      16,773
Maximum encounters for 1 patient: 40
Cleaned rows per patient (max):  1


## Missing values and quality problems
This cell shows the share and number of missing values in each column. It also counts unknown gender values and the share of patients with lab tests. It finds the columns that have one value in all rows, because these columns give no information.

In [45]:
miss = (raw.isna().mean()
        .loc[lambda s: s > 0]
        .sort_values(ascending=False)
        .to_frame("missing_share"))
miss["missing_count"] = raw[miss.index].isna().sum()
display(miss.style.format({"missing_share": "{:.1%}", "missing_count": "{:,}"}))

print("Unknown gender:", (raw["gender"] == "Unknown/Invalid").sum())
print("HbA1c tested:        ", f"{(raw['A1Cresult'] != 'None').mean():.1%}")
print("Glucose serum tested:", f"{(raw['max_glu_serum'] != 'None').mean():.1%}")
single = [c for c in raw.columns if raw[c].nunique(dropna=False) == 1]
print("Columns with one value:", single)

,missing_share,missing_count
weight,96.9%,"98,569"
medical_specialty,49.1%,"49,949"
payer_code,39.6%,"40,256"
race,2.2%,"2,273"
diag_3,1.4%,"1,423"
diag_2,0.4%,358
diag_1,0.0%,21


Unknown gender: 3
HbA1c tested:         16.7%
Glucose serum tested: 5.3%
Columns with one value: ['examide', 'citoglipton']


## Target
This cell compares the readmission classes in the raw data and in the cleaned data. Readmission in less than 30 days is the rare class. Thus, accuracy is not a good measure for the model.

In [46]:
target = pd.DataFrame({
    "raw":     raw["readmitted"].value_counts(normalize=True),
    "cleaned": df["readmitted"].value_counts(normalize=True),
}).loc[["NO", ">30", "<30"]]
display(target.style.format("{:.1%}"))
print(f"Cleaned 30-day readmissions: {df['readmitted_30d'].sum():,} "
      f"({df['readmitted_30d'].mean():.1%})")

,raw,cleaned
readmitted,,
NO,53.9%,59.3%
>30,34.9%,31.8%
<30,11.2%,9.0%


Cleaned 30-day readmissions: 6,285 (9.0%)


## Demographics
This cell shows the age and race of the patients. Most patients are 60 years or older, and most are Caucasian. These facts limit how well the results apply to other groups.

In [47]:
age60 = raw["age"].isin(["[60-70)", "[70-80)", "[80-90)", "[90-100)"]).mean()
print(f"Encounters for patients aged 60+: {age60:.1%}")
display(raw["race"].value_counts(normalize=True).to_frame("share").style.format("{:.1%}"))

Encounters for patients aged 60+: 67.4%


,share
race,
Caucasian,76.5%
AfricanAmerican,19.3%
Hispanic,2.0%
Other,1.5%
Asian,0.6%


## Check against published sources
This cell compares the notebook with values from the UCI dataset page [3] and from Strack et al. [5]. These sources are independent of this project. Small differences are possible, because Strack et al. used a different filter order and rounded their percentages.

In [48]:
final_a1c_tested = (df["A1Cresult"] != "Not_Tested").mean()

pub = pd.DataFrame([
    # check,                          notebook value,                   published, source
    ("Raw encounters",                len(raw),                         101_766, "UCI [3]; Strack [5] Methods"),
    ("Final encounters",              len(df),                          69_984,  "Strack [5] Methods"),
    ("30-day readmissions (final)",   df["readmitted_30d"].sum(),       6_459,   "Strack [5] Table 3"),
    ("HbA1c tested, final (%)",       round(final_a1c_tested*100, 1),   18.4,    "Strack [5] Results"),
    ("Weight missing (%)",            round(raw["weight"].isna().mean()*100, 1), 97,  "Strack [5] Table 1"),
    ("Payer code missing (%)",        round(raw["payer_code"].isna().mean()*100, 1), 40, "Strack [5] Methods"),
], columns=["check", "notebook", "published", "source"]).set_index("check")
pub["difference"] = pub["notebook"] - pub["published"]
display(pub)

,notebook,published,source,difference
check,,,,
Raw encounters,101766.0,101766.0,UCI [3]; Strack [5] Methods,0.0
Final encounters,69987.0,69984.0,Strack [5] Methods,3.0
30-day readmissions (final),6285.0,6459.0,Strack [5] Table 3,-174.0
"HbA1c tested, final (%)",18.4,18.4,Strack [5] Results,0.0
Weight missing (%),96.9,97.0,Strack [5] Table 1,-0.1
Payer code missing (%),39.6,40.0,Strack [5] Methods,-0.4


## Internal checks
This cell checks that the numbers agree with each other. It does not use the report or the published values. The removed rows must add up to the difference between the raw data and the cleaned data, each patient must have one row, and the classes must add up to the total.

A. Read the ID mappings

In [49]:
def read_ids_mapping(path):
    tables, current, name = {}, [], None
    for line in Path(path).read_text().splitlines():
        if not line.strip(",").strip():          # empty separator row
            if name: tables[name] = pd.DataFrame(current, columns=[name, "description"])
            current, name = [], None
        elif name is None:                       # header row
            name = line.split(",")[0]
        else:
            code, desc = line.split(",", 1)
            current.append((int(code), desc.strip().strip('"')))
    if name: tables[name] = pd.DataFrame(current, columns=[name, "description"])
    return tables

ids = read_ids_mapping(ROOT / "Data/DataMainTable/IDS_mapping.csv")
dd = ids["discharge_disposition_id"]
EXPIRED = dd.loc[dd["description"].str.startswith("Expired"), "discharge_disposition_id"].tolist()
HOSPICE = dd.loc[dd["description"].str.startswith("Hospice"), "discharge_disposition_id"].tolist()
print("Expired codes:", EXPIRED)   # expect [11, 19, 20, 21]
print("Hospice codes:", HOSPICE)   # expect [13, 14]

Expired codes: [11, 19, 20, 21]
Hospice codes: [13, 14]


B. Check

In [50]:
dead_hospice = raw["discharge_disposition_id"].isin(EXPIRED + HOSPICE)
after_dh   = raw[~dead_hospice]
after_first = after_dh.sort_values("encounter_id").drop_duplicates("patient_nbr")
after_gender = after_first[after_first["gender"] != "Unknown/Invalid"]

steps = pd.Series({
    "raw":                 len(raw),
    "after death/hospice": len(after_dh),
    "after first stay":    len(after_first),
    "after gender":        len(after_gender),
    "cleaned file":        len(df),
})
display(steps.to_frame("rows"))

assert len(after_gender) == len(df),              "Recalculated cohort differs from cleaned file"
assert df["patient_nbr"].is_unique,               "A patient has more than one row"
assert df["encounter_id"].is_unique,              "Duplicate encounter"
assert df["readmitted"].value_counts().sum() == len(df), "Target has missing values"
assert set(train.patient_nbr).isdisjoint(test.patient_nbr) if "train" in globals() else True
print("All internal checks passed")

,rows
raw,101766
after death/hospice,99343
after first stay,69990
after gender,69987
cleaned file,69987


All internal checks passed


## Summary of all checks
This cell collects the results of all checks in one table. Each check has a status:
- **PASS**: the value agrees with the rule.
- **FAIL**: the value does not agree with the rule.
- **INFO**: a known difference from the source study. It is not an error, but the report must explain it.

If one check has the status FAIL, the cell stops with an error.

In [51]:
rows = []
def check(section, name, value, rule, ok=None, info=False):
    status = "INFO" if info else ("PASS" if ok else "FAIL")
    rows.append({"section": section, "check": name, "value": value,
                 "rule": rule, "status": status})

# 1. Data structure
check("Structure", "Raw shape", f"{raw.shape[0]:,} × {raw.shape[1]}",
      "101,766 × 50", raw.shape == (101_766, 50))
check("Structure", "Cleaned shape", f"{df.shape[0]:,} × {df.shape[1]}",
      "69,987 × 35 (34 + target)", df.shape == (69_987, 35))

# 2. Unit of analysis
check("Patients", "Unique patients (raw)", f"{visits.size:,}",
      "fewer than encounters", visits.size < len(raw))
check("Patients", "Patients with >1 encounter", f"{(visits > 1).sum():,}",
      "> 0 (rows not independent)", (visits > 1).sum() > 0)
check("Patients", "One row per patient (cleaned)",
      int(df.groupby("patient_nbr").size().max()), "max = 1",
      df["patient_nbr"].is_unique)
check("Patients", "Unique encounter_id (cleaned)", df["encounter_id"].is_unique,
      "True", df["encounter_id"].is_unique)

# 3. Data quality
check("Quality", "'None' kept as text in A1Cresult", "A1Cresult" not in miss.index,
      "not in the missing-value list", "A1Cresult" not in miss.index)
check("Quality", "Single-value columns (raw)", ", ".join(single),
      "examide, citoglipton", set(single) == {"examide", "citoglipton"})
check("Quality", "Single-value columns removed (cleaned)",
      [c for c in single if c in df.columns], "none left",
      not any(c in df.columns for c in single))
check("Quality", "Unknown gender (cleaned)",
      int((df["gender"] == "Unknown/Invalid").sum()), "0",
      (df["gender"] == "Unknown/Invalid").sum() == 0)
check("Quality", "Target has no missing values", int(df["readmitted"].isna().sum()),
      "0", df["readmitted"].isna().sum() == 0)

# 4. Target
rate = df["readmitted_30d"].mean()
check("Target", "30-day readmission rate (cleaned)", f"{rate:.1%}",
      "rare class: 5–15%", 0.05 <= rate <= 0.15)

# 5. Published sources (UCI [3], Strack et al. [5])
check("Published", "Raw encounters vs UCI [3] / Strack [5]", f"{len(raw):,}",
      "101,766", len(raw) == 101_766)
check("Published", "HbA1c tested (final) vs Strack [5]", f"{final_a1c_tested:.1%}",
      "18.4% ± 0.5", abs(final_a1c_tested*100 - 18.4) <= 0.5)
check("Published", "Weight missing vs Strack [5]",
      f"{raw['weight'].isna().mean():.1%}", "97% ± 1",
      abs(raw["weight"].isna().mean()*100 - 97) <= 1)
check("Published", "Payer code missing vs Strack [5]",
      f"{raw['payer_code'].isna().mean():.1%}", "40% ± 1",
      abs(raw["payer_code"].isna().mean()*100 - 40) <= 1)
check("Published", "Final encounters vs Strack [5]",
      f"{len(df):,} ({len(df) - 69_984:+})", "69,984", info=True)
check("Published", "30-day readmissions vs Strack [5]",
      f"{df['readmitted_30d'].sum():,} ({df['readmitted_30d'].sum() - 6_459:+})",
      "6,459", info=True)

# 6. Recalculated cohort
check("Cohort", "Death/hospice codes from IDS_mapping",
      f"{EXPIRED} + {HOSPICE}", "[11, 19, 20, 21] + [13, 14]",
      EXPIRED == [11, 19, 20, 21] and HOSPICE == [13, 14])
check("Cohort", "Rows removed: death/hospice", f"{len(raw) - len(after_dh):,}",
      "2,423", len(raw) - len(after_dh) == 2_423)
check("Cohort", "Rows removed: not first stay",
      f"{len(after_dh) - len(after_first):,}", "29,353",
      len(after_dh) - len(after_first) == 29_353)
check("Cohort", "Rows removed: unknown gender",
      f"{len(after_first) - len(after_gender):,}", "3",
      len(after_first) - len(after_gender) == 3)
check("Cohort", "Recalculated cohort = cleaned file",
      f"{len(after_gender):,} vs {len(df):,}", "equal", len(after_gender) == len(df))
check("Cohort", "Same encounters as cleaned file",
      set(after_gender["encounter_id"]) == set(df["encounter_id"]), "True",
      set(after_gender["encounter_id"]) == set(df["encounter_id"]))

# 7. Train/test split (only if the split cell has run)
if "train" in globals():
    check("Split", "Train / test rows", f"{len(train):,} / {len(test):,}",
          "80% / 20%", abs(len(test)/len(df) - 0.2) < 0.01)
    check("Split", "Readmission rate train vs test",
          f"{train['readmitted_30d'].mean():.2%} / {test['readmitted_30d'].mean():.2%}",
          "difference < 0.5 pp",
          abs(train["readmitted_30d"].mean() - test["readmitted_30d"].mean()) < 0.005)
    check("Split", "Patients in both sets",
          len(set(train.patient_nbr) & set(test.patient_nbr)), "0",
          set(train.patient_nbr).isdisjoint(test.patient_nbr))
else:
    check("Split", "Train/test split", "not run", "run the split cell", ok=False)

summary = pd.DataFrame(rows)
display(summary)
counts = summary["status"].value_counts()
print(f"PASS: {counts.get('PASS', 0)}   INFO: {counts.get('INFO', 0)}   FAIL: {counts.get('FAIL', 0)}")
assert counts.get("FAIL", 0) == 0, "One or more checks failed. See the table above."
print("All checks passed.")

,section,check,value,rule,status
0,Structure,Raw shape,"101,766 × 50","101,766 × 50",PASS
1,Structure,Cleaned shape,"69,987 × 35","69,987 × 35 (34 + target)",PASS
2,Patients,Unique patients (raw),"71,518",fewer than encounters,PASS
3,Patients,Patients with >1 encounter,"16,773",> 0 (rows not independent),PASS
4,Patients,One row per patient (cleaned),1,max = 1,PASS
5,Patients,Unique encounter_id (cleaned),True,True,PASS
6,Quality,'None' kept as text in A1Cresult,True,not in the missing-value list,PASS
7,Quality,Single-value columns (raw),"examide, citoglipton","examide, citoglipton",PASS
8,Quality,Single-value columns removed (cleaned),[],none left,PASS
9,Quality,Unknown gender (cleaned),0,0,PASS


PASS: 25   INFO: 2   FAIL: 0
All checks passed.


## Findings
- The raw data has 101,766 encounters from 71,518 patients. Of these patients, 16,773 have more than one encounter. Thus, the cleaned data keeps one encounter for each patient.
- Weight (96.9%), medical specialty (49.1%) and payer code (39.6%) have the most missing values. In the lab columns, "None" means "not tested".
- The cleaned data has 69,987 encounters. Of these, 9.0% are readmissions in less than 30 days. The target is not balanced.
- The notebook agrees with Strack et al. [5] for HbA1c testing (18.4%), weight and payer code. The final cohort has 3 more encounters and 174 fewer readmissions than the source study. A possible cause is the sequence of the filters.
- The recalculated cohort from the raw data has the same encounters as the cleaned file. Thus, the preparation scripts follow their documented rules.